# Intraday Anomaly Repair for MatrixStoreMinuteFeed

This notebook repairs **persistent intraday price-level changes** inside the matrix store.

It is designed for cases where a symbol suddenly moves to a different scale during the day
and then **stays there**. Typical reasons include splits, bonus issues, or a vendor/broker
changing the price scale intraday.

---

## Important idea

Your simulator does **not** automatically adjust position quantities when a split happens.
So if the raw feed suddenly halves a held symbol's price, the synthetic NAV collapses even
though a real broker would have doubled your share count.

To keep the synthetic NAV continuous, this notebook repairs the **price path** instead:
it rescales the post-event suffix so the series stays on the same level.


## What the repairer does

It scans the store chronologically and, for each symbol:

1. compares a rolling median **before** a timestamp with a rolling median **after** it
2. checks whether the ratio looks like a canonical corporate-action factor such as
   `2`, `3`, `1/2`, `1/3`, `3/2`, etc.
3. if the new level persists, multiplies the **suffix** by a correction factor
4. carries that correction into all future dates for the same symbol

This makes the repaired store suitable for your simulator without changing position logic.


In [1]:
from pathlib import Path
import pandas as pd

from preprocess.intraday_anomaly_repair import repair_intraday_anomalies

INPUT_STORE = Path('E:/data_1m/processed_data/nifty500/1m_cube_store_repaired')
OUTPUT_STORE = Path('E:/data_1m/processed_data/nifty500/1m_cube_store_repaired_v2')

# Keep symbols=None to scan the whole store, or provide a list to focus on a few names.
SYMBOLS = None

PARAMS = {
    'lookback_bars': 20,
    'lookahead_bars': 20,
    'min_valid_window': 8,
    'min_jump_abs_return': 0.35,
    'factor_tolerance': 0.12,
}


## Run the repair pass

The output is a **new matrix store**. This is safer than modifying the current one in place.

The notebook also writes:

- `intraday_anomalies.csv`: one row per detected event
- `intraday_repair_manifest.json`: parameters and cumulative symbol multipliers


In [2]:
summary = repair_intraday_anomalies(
    input_store=INPUT_STORE,
    output_store=OUTPUT_STORE,
    symbols=SYMBOLS,
    **PARAMS,
)

print(summary)


RepairSummary(output_store='E:\\data_1m\\processed_data\\nifty500\\1m_cube_store_repaired_v2', scanned_dates=2587, anomalies_found=742, symbols_touched=88, manifest_path='E:\\data_1m\\processed_data\\nifty500\\1m_cube_store_repaired_v2\\intraday_repair_manifest.json', anomalies_csv='E:\\data_1m\\processed_data\\nifty500\\1m_cube_store_repaired_v2\\intraday_anomalies.csv')


## Inspect the anomalies that were found

The `correction_applied_to_suffix` column is the most important one.

Example:
- if raw prices drop from ~100 to ~50 and stay there,
  `correction_applied_to_suffix` will be close to `2.0`
- this means the repaired store multiplies the post-event suffix by `2.0`


In [ ]:
anomalies_path = Path(summary.anomalies_csv)
anomalies = pd.read_csv(anomalies_path)
anomalies.sort_values(['date', 'symbol', 'ts']).head(50)


## Spot-check a repaired symbol visually

Replace `SYMBOL_TO_CHECK` and `DATE_TO_CHECK` with a suspicious event from the anomaly table.
This cell compares the raw and repaired series for a single day.


In [ ]:
SYMBOL_TO_CHECK = 'CHANGE_ME_SYMBOL'
DATE_TO_CHECK = 'YYYY-MM-DD'

raw_day = pd.read_parquet(INPUT_STORE / f'date={DATE_TO_CHECK}' / 'close.parquet')
repaired_day = pd.read_parquet(OUTPUT_STORE / f'date={DATE_TO_CHECK}' / 'close.parquet')

raw_day['ts'] = pd.to_datetime(raw_day['ts'])
repaired_day['ts'] = pd.to_datetime(repaired_day['ts'])

compare = pd.DataFrame({
    'ts': raw_day['ts'],
    'raw': raw_day[SYMBOL_TO_CHECK],
    'repaired': repaired_day[SYMBOL_TO_CHECK],
})
compare.plot(x='ts', y=['raw', 'repaired'], figsize=(14, 5), title=f'{SYMBOL_TO_CHECK} on {DATE_TO_CHECK}')


## After the repair

Point your run config to `OUTPUT_STORE`, rerun the strategy, and then use the
**NAV Spike Auditor** notebook. That is the fastest way to verify whether the large
cliffs are gone and which symbols still need attention.
